In [ ]:
# Version 1 - For final run - JPK 2026-08-06

# Quality Check insitu data and filter stations before use

# Check 1 - Data sufficiency - 80% data coverage for the study period (2001-01-01 to 2025-12-31)
# Check 2 - Outlier - check tmax>tmin for same day-station,
#                       flag values > 99.9 percentile & < 0.1 percentile; followed by manual check
# Check 3 - Frozen sensor - same value for more than 8 days - flagged and manually checked
# Check 4 - Temporal Consistency - Flag A. Observations with diff +- 3 sd from climatological monthly mean 
#                                       B. spike detection using scipy find_peaks
# Check 5 - Location - Check station locations are valid
# Check 6 - Spatial Homogeneity - DMCA
# Check 7 - Station Relocation - PMFT


In [ ]:
# Libraries
import os
import numpy as np  
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.spatial import cKDTree
import geopandas as gpd
from shapely.geometry import Point
import geodatasets
import piecewise_regression

# read insitu data
tmax_raw = pd.read_csv(fr"..\Data_Temp\Insitu\Raw\tmax_01_25.csv")
tmin_raw = pd.read_csv(fr"..\Data_Temp\Insitu\Raw\tmin_01_25.csv")
metadata = pd.read_csv(fr"..\Data_Temp\Insitu\Raw\station_metadata_final.csv")

# temp folder
temporary_folder = r"C:\Users\A S U S\Documents\Study\ClimateData\CD-Workspace\Data_Temp\Insitu\Junk"
os.makedirs(temporary_folder, exist_ok=True)

start_date = pd.Timestamp("2001-01-01")
end_date = pd.Timestamp("2025-12-31")
total_days = (end_date - start_date).days + 1

In [ ]:
# Check 1 - Data sufficiency - 80% data coverage for the study period (2001-01-01 to 2025-12-31)
# additional check of removing duplicates is performed before anything


tmax_raw['date'] = pd.to_datetime(tmax_raw['date'])
tmin_raw['date'] = pd.to_datetime(tmin_raw['date'])

tmax_dup = tmax_raw[tmax_raw.duplicated(subset=['date', 'index_no'], keep=False)]
tmax = tmax_raw[~tmax_raw.index.isin(tmax_dup.index)]

tmin_dup = tmin_raw[tmin_raw.duplicated(subset=['date', 'index_no'], keep=False)]
tmin = tmin_raw[~tmin_raw.index.isin(tmin_dup.index)]

tmax_coverage = tmax.groupby('index_no').agg(
    first_date = ('date', 'min'),
    last_date =  ('date', 'max'),
    total_records = ('date', 'count'),
    missing_percentage=('max_temp', lambda x: (total_days - x.count()) / total_days * 100)
).reset_index()

tmin_coverage = tmin.groupby('index_no').agg(
    first_date = ('date', 'min'),
    last_date =  ('date', 'max'),
    total_records = ('date', 'count'),
    missing_percentage=('min_temp', lambda x: (total_days - x.count()) / total_days * 100)
).reset_index()

tmax_qualify_list = list((tmax_coverage[tmax_coverage['missing_percentage'] <= 20])['index_no'])
tmax_qc1 = tmax[tmax['index_no'].isin(tmax_qualify_list)]

tmin_qualify_list = list((tmin_coverage[tmin_coverage['missing_percentage'] <= 20])['index_no'])
tmin_qc1 = tmin[tmin['index_no'].isin(tmin_qualify_list)]

print(f"Tmax: {len(tmax_qc1['index_no'].unique())} out of stations qualify out of {len(tmax['index_no'].unique())}")
print(f"Tmax: {len(tmin_qc1['index_no'].unique())} out of stations qualify out of {len(tmin['index_no'].unique())}")

print("\nTmax: Data dropped from QC1:", f"{((len(tmax_raw) - len(tmax_qc1))/ len(tmax_raw)) * 100:.2f}", "%")
print("Tmin: Data dropped from QC1:", f"{((len(tmin_raw) - len(tmin_qc1))/ len(tmin_raw)) * 100:.2f}", "%")

In [ ]:
# Check 2 - Outlier - check tmax > tmin for same day-station,
#                       flag values > 99.9 percentile & < 0.1 percentile; followed by manual check

temp_merged = tmax_qc1.merge(tmin_qc1, how= 'inner', on = ['index_no', 'year', 'month', 'day']).drop(columns= ['date_x', 'date_y'])
temp_merged_dif = temp_merged['max_temp'] - temp_merged['min_temp']
temp_merged['diff'] = temp_merged_dif

temp_merged_error = temp_merged[temp_merged['diff'] <= 0]
error_combinations_np = temp_merged_error[['index_no', 'year', 'month', 'day']].to_numpy()
error_combinations_tuple = [tuple(row) for row in error_combinations_np]

tmax_qc2a = tmax_qc1[~tmax_qc1[['index_no', 'year', 'month', 'day']].apply(tuple, axis=1).isin(error_combinations_tuple)]
tmin_qc2a = tmin_qc1[~tmin_qc1[['index_no', 'year', 'month', 'day']].apply(tuple, axis=1).isin(error_combinations_tuple)]

tmax_extreme = tmax_qc2a[(tmax_qc2a['max_temp'] > np.percentile(tmax_qc2a['max_temp'].to_numpy(), 99.9)) | (tmax_qc2a['max_temp'] < np.percentile(tmax_qc2a['max_temp'].to_numpy(), 0.1))]
tmin_extreme = tmin_qc2a[(tmin_qc2a['min_temp'] > np.percentile(tmin_qc2a['min_temp'].to_numpy(), 99.9)) | (tmin_qc2a['min_temp'] < np.percentile(tmin_qc2a['min_temp'].to_numpy(), 0.1))]

# #pd index of extreme values to remove
# for tmax none required removal
print(f"Tmax range: [{min(list(tmax_extreme['max_temp']))}, {max(list(tmax_extreme['max_temp']))}]")
print(f"Tmin range: [{min(list(tmin_extreme['min_temp']))}, {max(list(tmin_extreme['min_temp']))}]")

tmax_qc2 = tmax_qc2a
tmin_qc2 = tmin_qc2a

In [ ]:
# Check 3 - Frozen sensor - same value for more than 8 days - flagged and manually checked

tmax_qc2_idx = list(tmax_qc2['index_no'].unique())

frozen_stations_df_tmax = pd.DataFrame()

for stn in tmax_qc2_idx:
    frozen_date_stn = {"index_no": stn}
    tmax_station = tmax_qc2[tmax_qc2['index_no'] == stn].copy()
    
    roll_max = tmax_station['max_temp'].rolling(10).max()
    roll_min = tmax_station['max_temp'].rolling(10).min()
    
    mask = ((roll_max == roll_min) & roll_max.notna()).shift(-9, fill_value=False)
    
    frozen_date_stn['dates'] = tmax_station[mask]['date'].tolist()
    frozen_date_stn = pd.DataFrame([frozen_date_stn])
    frozen_stations_df_tmax = pd.concat([frozen_stations_df_tmax, frozen_date_stn], ignore_index=True)

# manual check for tmax -  no values were removed - repeated values were within plausible range based on values from dates before and after repitition

tmin_qc2_idx = list(tmin_qc2['index_no'].unique())

frozen_stations_df_tmin = pd.DataFrame()

for stn in tmin_qc2_idx:
    frozen_date_stn = {"index_no": stn}
    tmin_station = tmin_qc2[tmin_qc2['index_no'] == stn].copy()
    
    roll_max = tmin_station['min_temp'].rolling(10).max()
    roll_min = tmin_station['min_temp'].rolling(10).min()
    
    mask = ((roll_max == roll_min) & roll_max.notna()).shift(-9, fill_value=False)
    
    frozen_date_stn['dates'] = tmin_station[mask]['date'].tolist()
    frozen_date_stn = pd.DataFrame([frozen_date_stn])
    frozen_stations_df_tmin = pd.concat([frozen_stations_df_tmin, frozen_date_stn], ignore_index=True)

# manual check for tmax -  no values were removed - repeated values were within plausible range based on values from dates before and after repitition

# save intermediate file to check and remove from the next step 
tmax_qc2.to_csv(os.path.join(temporary_folder, "tmax_qc2.csv"), index = False)
tmin_qc2.to_csv(os.path.join(temporary_folder, "tmin_qc2.csv"), index = False)

In [ ]:
# Check 4 - Temporal Consistency - Flag A. Observations with diff +- 5 sd from climatological monthly mean 
#                                       B. spike detection using scipy find_peaks
# double flagged are removed

tmax_clim = tmax_qc2.groupby(['index_no', 'month']).agg(
    clim_mean = ('max_temp', 'mean')
)
tmax_qc3a = tmax_qc2.merge(tmax_clim, how = 'left', on = ['index_no', 'month'])
tmax_qc3a['t_diff'] = tmax_qc3a['max_temp'] - tmax_qc3a['clim_mean']

tmax_qc3a_upper = np.mean(tmax_qc3a['t_diff'].to_numpy()) + 5 * np.std(tmax_qc3a['t_diff'].to_numpy())
tmax_qc3a['flag_A'] = (tmax_qc3a['t_diff'] > tmax_qc3a_upper) | (tmax_qc3a['t_diff'] < -tmax_qc3a_upper)
tmax_qc3a_inspect = tmax_qc3a[tmax_qc3a['flag_A'] == True].sort_values(['index_no','date']).reset_index(drop=True)


tmin_clim = tmin_qc2.groupby(['index_no', 'month']).agg(
    clim_mean = ('min_temp', 'mean')
)
tmin_qc3a = tmin_qc2.merge(tmin_clim, how = 'left', on = ['index_no', 'month'])
tmin_qc3a['t_diff'] = tmin_qc3a['min_temp'] - tmin_qc3a['clim_mean']

tmin_qc3a_upper = np.mean(tmin_qc3a['t_diff'].to_numpy()) + 5 * np.std(tmin_qc3a['t_diff'].to_numpy())
tmin_qc3a['flag_A'] = (tmin_qc3a['t_diff'] > tmin_qc3a_upper) | (tmin_qc3a['t_diff'] < -tmin_qc3a_upper)
tmin_qc3a_inspect = tmin_qc3a[tmin_qc3a['flag_A'] == True].sort_values(['index_no','date']).reset_index(drop=True)


def flag_spikes(df, value_col, group_col='index_no', k = 5,
                 max_width_days=2, wlen_days=21, min_diffs=30):
    """
    Flag B: per-station spurious-spike/dip detection on a deseasonalized
    residual series (e.g. t_diff). Requires df to have [group_col, 'date', value_col].
    """
    df = df.sort_values([group_col, 'date']).copy()
    df['flag_B'] = False

    for stn, g in df.groupby(group_col):
        g = g.set_index('date')
        full_idx = pd.date_range(g.index.min(), g.index.max(), freq='D')
        g = g.reindex(full_idx)
        x = g[value_col].to_numpy()

        # Robust, station-specific scale from the first-difference distribution
        first_diff = np.diff(x)
        valid_diff = first_diff[~np.isnan(first_diff)]
        if len(valid_diff) < min_diffs:      # same 30-sample floor as your variogram work
            continue
        mad = np.median(np.abs(valid_diff - np.median(valid_diff)))
        thresh = k * mad
        if thresh == 0:
            continue

        peak_idx = set()
        for series in (x, -x):               # upward spikes, then downward dips
            idx, _ = find_peaks(
                series,
                threshold=thresh,
                width=(None, max_width_days),
                wlen=wlen_days,
            )
            peak_idx.update(idx)

        confirmed = []
        for i in peak_idx:
            if i == 0 or i == len(x) - 1:
                continue
            if np.isnan(x[i - 1]) or np.isnan(x[i + 1]):
                continue                      # can't confirm "vs. neighbour" across a data gap
            if abs(x[i - 1] - x[i + 1]) < thresh / 2:
                confirmed.append(i)

        flagged_dates = g.index[confirmed]
        df.loc[(df[group_col] == stn) & (df['date'].isin(flagged_dates)), 'flag_B'] = True

    return df

tmax_qc3a = flag_spikes(tmax_qc3a, value_col='t_diff')
tmin_qc3a = flag_spikes(tmin_qc3a, value_col='t_diff')

tmax_qc3a_flagged = tmax_qc3a[(tmax_qc3a['flag_A'] == True) & (tmax_qc3a['flag_B'] == True)]
tmin_qc3a_flagged = tmin_qc3a[(tmin_qc3a['flag_A'] == True) & (tmin_qc3a['flag_B'] == True)]


# tmax qc3 - algorithm flagged 17 values out of which 2 were removed following manual check
# three clues for manual check : 1. does the 30 day period include another value close to the flagged  
# then likely not an error but an expected extreme event (up to 5 degrees)
# 2. is the mistake likely digit record error
# ie instead of 11.1 , 1.1 is recorded or 21.5 is mistakely taken as 1.5, this is checked by reading nearby records
# 3. is the average value for the 30 day period within ~12-15 degrees
# manual check follows a more conservative approach, unless sufficiently different, extreme values are kept

# for tmin - algorithm flagged 25 values, 14 observations were removed following manual check

# remove tmax values of index: 329239, 217898
# remove tmin values of index: 371386, 279378, 121562, 169381, 328953, 225066, 264112, 28742, 264994, 294669, 355724, 325537
# 300548, 183828 
tmax_idx_remove = [329239, 217898]
tmin_idx_remove = [371386, 279378, 121562, 169381, 328953, 225066, 264112, 28742, 264994, 294669, 355724, 325537, 300548, 183828]

tmax_qc3 = (tmax_qc3a[~tmax_qc3a.index.isin(tmax_idx_remove)].reset_index(drop=True))[['date', 'index_no', 'year', 'month', 'day', 'max_temp']]
tmin_qc3 = tmin_qc3a[~tmin_qc3a.index.isin(tmin_idx_remove)].reset_index(drop=True)[['date', 'index_no', 'year', 'month', 'day', 'min_temp']]

print("\nTmax: Data dropped from QC1:", f"{((len(tmax_qc2) - len(tmax_qc3)))}")
print("Tmin: Data dropped from QC1:", f"{((len(tmin_qc2) - len(tmin_qc3)))}")

In [ ]:
# Check 5 - Location - Check station locations are valid
# Check 6 - Spatial Homogeneity - DMCA

# A. for each station find all stations within 100km radius
# B. calculate the running cumulative sum of daily tmax/tmin values for the station (y) vs the same for the average of the other stations (x)
# C. detect break points (at p< 0.05)


def latlon_to_3d(lat_deg, lon_deg):
    """Converts arrays of latitude and longitude (degrees) to 3D Cartesian units."""
    lat_rad = np.radians(lat_deg)
    lon_rad = np.radians(lon_deg)
    
    x = np.cos(lat_rad) * np.cos(lon_rad)
    y = np.cos(lat_rad) * np.sin(lon_rad)
    z = np.sin(lat_rad)
    
    return np.column_stack((x, y, z))

def find_neighbor_stations(metadata_df, max_dist_km=100.0, max_elev_diff_m=1000.0):
    """
    Finds all neighboring station index_no's within max_dist_km and max_elev_diff_m
    for each station in metadata_df.
    
    Parameters:
    - metadata_df: DataFrame containing ['index_no', 'x' (lon), 'y' (lat), 'elev' (elevation in meters)]
    - max_dist_km: Maximum horizontal distance radius in kilometers
    - max_elev_diff_m: Maximum elevation difference in meters (default 1000 m)
    
    Returns:
    - Dict[str/int, List[str/int]]: Dictionary mapping target index_no -> list of neighbor index_no's
    """
    EARTH_RADIUS_KM = 6371.0
    max_dist_rad = max_dist_km / EARTH_RADIUS_KM
    chord_threshold = 2 * np.sin(max_dist_rad / 2)
    
    # Note: latlon_to_3d expects (lat, lon) -> (y, x)
    coords_3d = latlon_to_3d(metadata_df['y'].values, metadata_df['x'].values)
    elevations = metadata_df['elev'].values
    
    tree = cKDTree(coords_3d)
    matched_indices = tree.query_ball_point(coords_3d, r=chord_threshold)
    
    stations = metadata_df['index_no'].values
    neighbors_dict = {}
    
    for i, matches in enumerate(matched_indices):
        target_stn = stations[i]
        target_elev = elevations[i]
        # Exclude self match and apply elevation difference criterion
        neighbor_stns = [
            stations[idx] for idx in matches
            if idx != i and abs(int(elevations[idx]) - int(target_elev)) <= max_elev_diff_m
        ]
        neighbors_dict[target_stn] = neighbor_stns
        
    return neighbors_dict


# Prepare dictionary of matched stations for tmax
tmax_stations = (tmax_qc3['index_no'].unique().astype(str))
tmax_metadata = metadata[metadata['Index No.'].isin(tmax_stations)]
tmax_metadata = tmax_metadata[['Index No.', 'Lon', 'Lat', 'Elev']].rename(columns = {'Index No.': 'index_no',
                                                                                    'Lon': 'x',
                                                                                    'Lat': 'y',
                                                                                    'Elev': 'elev'}).reset_index(drop=True)
tmax_matched_stn = find_neighbor_stations(tmax_metadata)

# Prepare dictionary of matched stations for tmin
tmin_stations = (tmin_qc3['index_no'].unique().astype(str))
tmin_metadata = metadata[metadata['Index No.'].isin(tmin_stations)]
tmin_metadata = tmin_metadata[['Index No.', 'Lon', 'Lat', 'Elev']].rename(columns = {'Index No.': 'index_no',
                                                                                    'Lon': 'x',
                                                                                    'Lat': 'y',
                                                                                    'Elev': 'elev'}).reset_index(drop=True)
tmin_matched_stn = find_neighbor_stations(tmin_metadata)


In [ ]:
# Check 5 - Location - Check station locations are valid

geo_tmax = gpd.GeoDataFrame(
    tmax_metadata, 
    geometry=gpd.points_from_xy(tmax_metadata['x'], tmax_metadata['y']), 
    crs="EPSG:4326"
)
geo_tmin = gpd.GeoDataFrame(
    tmin_metadata, 
    geometry=gpd.points_from_xy(tmin_metadata['x'], tmin_metadata['y']), 
    crs="EPSG:4326"
)

world = gpd.read_file(geodatasets.get_path("naturalearth.land"))


fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10), sharex=True, sharey=True)

# Top Subplot - Tmax Stations
world.plot(ax=ax1, color='lightgrey', edgecolor='black')
geo_tmax.plot(ax=ax1, color='red', markersize=30, marker='o', label='Tmax Stations')
ax1.set_title("Tmax Station Locations")
ax1.legend(loc='upper right')

# Bottom Subplot - Tmin Stations
world.plot(ax=ax2, color='lightgrey', edgecolor='black')
geo_tmin.plot(ax=ax2, color='blue', markersize=30, marker='o', label='Tmin Stations')
ax2.set_title("Tmin Station Locations")
ax2.legend(loc='upper right')

ax1.set_xlim([80, 89])
ax1.set_ylim([26, 31])

plt.tight_layout()
plt.show()


In [ ]:
# %%
# Check 6 - Spatial Homogeneity - DMCA
# Visual double-mass inspection: flag sustained (~1yr) daily-bias deviations
# > 1 degC from a station's OWN long-term average bias vs its reference, mark
# them on the double-mass curve for manual review, and tally a count per station.
#
# NOTES:
# - davies_test / beta1-CI significance dropped in favour of manual visual review
#   feeding a confirmatory test (Check 7) on flagged stations only.
# - reference_average still uses median (not mean) - one bad neighbour distorting
#   everyone else's reference is a separate problem this doesn't fix (see clustering
#   discussion) - a flagged station whose neighbours are ALSO flagged in the same
#   window is still a contamination signature worth checking by eye across plots.
# - because deviation is measured against each station's OWN whole-record average
#   bias, a constant elevation-driven offset does NOT by itself trigger a flag -
#   only a CHANGE in that offset does.
# - target_col is now a parameter (not hardcoded 'target_tmax') so this same
#   function is reused for tmin without mislabeling the column.

DEVIATION_THRESHOLD_C = 2.0     # deg C
ROLLING_WINDOW_DAYS   = 365     # ~1yr - persistence is enforced by smoothing, not a separate rule
MIN_EVENT_DAYS        = 180     # drop flagged runs shorter than this (guards against a
                                 # borderline threshold-crossing counting as a whole event)
FIXED_YLIM            = (-4, 4) # identical y-scale on every station's bias panel - this is
                                 # what actually "standardizes the scale" across stations,
                                 # since the top (cumsum) panel's natural scale differs by
                                 # station record length and can't be forced comparable

def flag_and_plot_dmca(clean_df, target_stn_idx, target_col, save_dir=None, show=True):
    cumsum_col = f'{target_col}_cumsum'
    df = clean_df.copy()
    df['daily_bias'] = df[target_col] - df['reference_average']
    baseline = df['daily_bias'].mean()

    # rolling mean must respect actual calendar days, not row count - with up to 20%
    # missing days allowed by Check 1, a plain .rolling(365) on the gappy index would
    # span more than 365 calendar days. Reindex to continuous daily first (same fix
    # already used in flag_spikes, Check 4), then map back onto the real dates.
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq='D')
    rolling_full = df['daily_bias'].reindex(full_idx).rolling(
        ROLLING_WINDOW_DAYS, min_periods=int(ROLLING_WINDOW_DAYS * 0.8), center=True
    ).mean()
    df['rolling_bias'] = rolling_full.reindex(df.index)

    df['deviation'] = df['rolling_bias'] - baseline
    df['flagged'] = df['deviation'].abs() > DEVIATION_THRESHOLD_C

    # collapse consecutive flagged days into discrete events, measured in calendar days
    df['event_id'] = (df['flagged'] != df['flagged'].shift()).cumsum()
    events = []
    for eid, grp in df[df['flagged']].groupby('event_id'):
        span_days = (grp.index.max() - grp.index.min()).days + 1
        if span_days >= MIN_EVENT_DAYS:
            events.append({
                'index_no': target_stn_idx,
                'start_date': grp.index.min().date(),
                'end_date': grp.index.max().date(),
                'duration_days': span_days,
                'mean_deviation_C': round(grp['deviation'].mean(), 2),
                'direction': 'warm shift' if grp['deviation'].mean() > 0 else 'cool shift',
            })

    kept_dates = pd.DatetimeIndex([])
    if events:
        kept_dates = pd.DatetimeIndex(sorted(set().union(*[
            set(pd.date_range(e['start_date'], e['end_date'])) for e in events
        ])))
    df['marked'] = df.index.isin(kept_dates)

    # single-line OLS reference on the double-mass panel - visual aid only, not a test
    slope, intercept = np.polyfit(df['ref_avg_cumsum'], df[cumsum_col], 1)
    fit_line = slope * df['ref_avg_cumsum'] + intercept

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

    # Panel 1: classic double-mass curve + straight-line reference + flagged points
    ax1.plot(df['ref_avg_cumsum'], df[cumsum_col], color='steelblue', lw=1.2, label='DMC curve')
    ax1.plot(df['ref_avg_cumsum'], fit_line, color='grey', ls='--', lw=1, label='single-line reference')
    marked = df[df['marked']]
    if len(marked):
        ax1.scatter(marked['ref_avg_cumsum'], marked[cumsum_col],
                    facecolors='none', edgecolors='red', s=25, linewidths=1.2,
                    label='flagged deviation', zorder=5)
    ax1.set_xlabel('Reference cumulative sum (°C)')
    ax1.set_ylabel('Target cumulative sum (°C)')
    ax1.set_title(f'Station {target_stn_idx} — double-mass curve ({target_col})')
    ax1.legend(loc='upper left', fontsize=8)

    # Panel 2: daily bias + rolling mean, FIXED y-scale, threshold guides, flagged spans marked
    ax2.plot(df.index, df['daily_bias'], color='lightgrey', lw=0.4, label='daily bias (raw)')
    ax2.plot(df.index, df['rolling_bias'], color='black', lw=1.2, label=f'{ROLLING_WINDOW_DAYS}d rolling mean')
    ax2.axhline(baseline, color='grey', ls=':', lw=1)
    ax2.axhline(baseline + DEVIATION_THRESHOLD_C, color='red', ls='--', lw=0.8)
    ax2.axhline(baseline - DEVIATION_THRESHOLD_C, color='red', ls='--', lw=0.8)
    if len(marked):
        ax2.scatter(marked.index, marked['rolling_bias'],
                    facecolors='none', edgecolors='red', s=18, linewidths=1.0, zorder=5)
    ax2.set_ylim(*FIXED_YLIM)
    ax2.set_xlabel('Date')
    ax2.set_ylabel('Target − reference (°C)')
    ax2.set_title(f'{len(events)} flagged deviation(s) ≥{DEVIATION_THRESHOLD_C}°C, ≥{MIN_EVENT_DAYS}d')
    ax2.legend(loc='upper left', fontsize=8)

    plt.tight_layout()
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        fig.savefig(os.path.join(save_dir, f'DMCA_{target_col}_{target_stn_idx}.png'), dpi=120)
    if show:
        plt.show()
    else:
        plt.close(fig)

    return {'events': events, 'daily_bias': df['daily_bias'], 'baseline': baseline}

# Run over all tmax stations. Saves one PNG per station to temporary_folder for a
# quick flip-through review; set show=False if inline rendering for ~90+ stations
# gets slow.

dmca_visual_events = []
dmca_bias_series_tmax = {}    # station -> full daily_bias series, kept for Check 7

for target_stn_idx in tmax_stations:
    matched_stn_idx = tmax_matched_stn.get(target_stn_idx, [])
    if not matched_stn_idx:
        continue

    target_stn_df = tmax_qc3[tmax_qc3['index_no'] == int(target_stn_idx)].drop(columns=['date', 'index_no'])
    target_stn_df = target_stn_df.rename(columns={'max_temp': 'target_tmax'})

    dmca_stn = target_stn_df.copy()
    for i in matched_stn_idx:
        matched_stn_df = tmax_qc3[tmax_qc3['index_no'] == int(i)][['year', 'month', 'day', 'max_temp']]
        matched_stn_df = matched_stn_df.rename(columns={'max_temp': f'tmax_{i}'})
        dmca_stn = dmca_stn.merge(matched_stn_df, on=["year", "month", "day"], how='left')

    dmca_stn['date'] = pd.to_datetime(dmca_stn[['year', 'month', 'day']])
    dmca_stn = dmca_stn.set_index('date')

    neighbor_cols = [c for c in dmca_stn.columns if c.startswith("tmax_")]
    dmca_stn['reference_average'] = dmca_stn[neighbor_cols].median(axis=1)

    clean_df = dmca_stn[['target_tmax', 'reference_average']].dropna().copy()
    if len(clean_df) < ROLLING_WINDOW_DAYS:   # can't judge a ~1yr shift with less than a year of overlap
        print(f"Tmax station {target_stn_idx}: skipped, only {len(clean_df)} overlapping days")
        continue

    clean_df['target_tmax_cumsum'] = clean_df['target_tmax'].cumsum()
    clean_df['ref_avg_cumsum'] = clean_df['reference_average'].cumsum()

#    print(f'\nDMC Plot for tmax station {target_stn_idx} =============')
    result = flag_and_plot_dmca(clean_df, target_stn_idx, target_col='target_tmax',
                                 save_dir=temporary_folder, show=False)
    dmca_visual_events.extend(result['events'])
    dmca_bias_series_tmax[target_stn_idx] = result['daily_bias']



In [1]:
# Same pass over tmin stations - identical logic, tmin_* variables and 'min_temp'.
dmca_bias_series_tmin = {}

for target_stn_idx in tmin_stations:
    matched_stn_idx = tmin_matched_stn.get(target_stn_idx, [])
    if not matched_stn_idx:
        continue

    target_stn_df = tmin_qc3[tmin_qc3['index_no'] == int(target_stn_idx)].drop(columns=['date', 'index_no'])
    target_stn_df = target_stn_df.rename(columns={'min_temp': 'target_tmin'})

    dmca_stn = target_stn_df.copy()
    for i in matched_stn_idx:
        matched_stn_df = tmin_qc3[tmin_qc3['index_no'] == int(i)][['year', 'month', 'day', 'min_temp']]
        matched_stn_df = matched_stn_df.rename(columns={'min_temp': f'tmin_{i}'})
        dmca_stn = dmca_stn.merge(matched_stn_df, on=["year", "month", "day"], how='left')

    dmca_stn['date'] = pd.to_datetime(dmca_stn[['year', 'month', 'day']])
    dmca_stn = dmca_stn.set_index('date')

    neighbor_cols = [c for c in dmca_stn.columns if c.startswith("tmin_")]
    dmca_stn['reference_average'] = dmca_stn[neighbor_cols].median(axis=1)

    clean_df = dmca_stn[['target_tmin', 'reference_average']].dropna().copy()
    if len(clean_df) < ROLLING_WINDOW_DAYS:
        print(f"Tmin station {target_stn_idx}: skipped, only {len(clean_df)} overlapping days")
        continue

    clean_df['target_tmin_cumsum'] = clean_df['target_tmin'].cumsum()
    clean_df['ref_avg_cumsum'] = clean_df['reference_average'].cumsum()

#    print(f'\nDMC Plot for tmin station {target_stn_idx} =============')
    result = flag_and_plot_dmca(clean_df, target_stn_idx, target_col='target_tmin',
                                 save_dir=temporary_folder, show=False)
    dmca_visual_events.extend(result['events'])
    dmca_bias_series_tmin[target_stn_idx] = result['daily_bias']


# Summary table - candidate list to cross-check against Check 7

dmca_visual_summary = (pd.DataFrame(dmca_visual_events)
                        .sort_values(['index_no', 'start_date'])
                        .reset_index(drop=True)) if dmca_visual_events else pd.DataFrame()

n_flagged_stations = dmca_visual_summary['index_no'].nunique() if len(dmca_visual_summary) else 0
print(f"\n{n_flagged_stations} stations with at least one flagged deviation, "
      f"{len(dmca_visual_summary)} events total")
dmca_visual_summary

NameError: name 'tmin_stations' is not defined

* stations flagged for tmax: [605, 922, 1024]
* stations flagged for tmin: [105, 207, 409, 1419]
* manual check involved two criteria applied either: 1. consistent shift -- shift that doesnot return back to normal 
* 2. Heavy shift - prolonged shift > 4 degrees

In [ ]:
# %%
# Check 7 - Confirmatory homogeneity test (Pettitt + SNHT, run on ALL stations,
# not just the visually-flagged subset - a fixed threshold+duration filter is
# structurally blind to small breaks, short breaks, breaks at the record edges,
# and offsetting breaks, so it needs an independent check rather than only
# auditing what it already flagged).
#
# NOTE on method: PMFT (Wang 2008) is the test named in the original outline, but
# its only validated implementation is Environment Canada's RHtestsV4 - an R
# toolkit that isn't a single function call. It's file-based (specific date format,
# an explicit missing-value code) and semi-manual: call FindU.wRef, inspect the
# output, then re-run StepSize.wRef, repeating "until each and every changepoint
# retained is determined to be significant" per Wang & Feng's own manual - no
# batch mode. Re-deriving PMFT's statistic from scratch isn't a safe shortcut
# either - its penalty function and critical values come from Wang (2008)'s own
# Monte Carlo fitting, and getting that calibration even slightly wrong would
# produce a p-value that looks legitimate but isn't.
#
# Used instead: Pettitt's test and SNHT (Alexandersson 1986) via pyhomogeneity -
# validated, pure-Python, single-call, Monte-Carlo-calibrated p-values, and an
# accepted member of the same homogeneity-test family PMFT belongs to.
#
# pip install pyhomogeneity

import pyhomogeneity as hg

def run_homogeneity_tests(bias_series_dict, variable_label, alpha=0.05):
    rows = []
    for stn, bias in bias_series_dict.items():
        series = bias.dropna()
        if len(series) < 30:
            continue

        try:
            pettitt = hg.pettitt_test(series.to_numpy(), alpha=alpha)
        except Exception as e:
            print(f"Pettitt test failed for {variable_label} station {stn}: {e}")
            continue

        rows.append({
            'index_no': stn,
            'variable': variable_label,
            'n_obs': len(series),
            'significant': bool(pettitt.h),
            'breakpoint': series.index[pettitt.cp].date(),
            'p_value': round(float(pettitt.p), 4),
            'bias_shift_C': round(float(pettitt.avg.mu2 - pettitt.avg.mu1), 2),
        })

    return pd.DataFrame(rows)


pmft_summary_tmax = run_homogeneity_tests(dmca_bias_series_tmax, 'tmax')
pmft_summary_tmin = run_homogeneity_tests(dmca_bias_series_tmin, 'tmin')
pmft_summary = pd.concat([pmft_summary_tmax, pmft_summary_tmin], ignore_index=True)
pmft_summary

: 

In [ ]:
# %%
# Decision tree: TRIM only if confirmed unstable by both flags AND isolated
# from any neighbor showing an automated DMCA deviation (not just neighbors
# that also passed Pettitt) - conservative on the contamination side, since a
# neighbor's blip is treated as possible shared-cause evidence even if a human
# later judged it not worth flagging on its own merits.
# Everything else -> KEEP entire station record intact.

tmax_flagged = ['605', '922', '1024']
tmin_flagged = ['105', '207', '409', '1419']

def build_removal_plan(flagged_list, matched_stn_dict, visual_summary, pmft_summary, variable_label):
    # Step 1: which flagged stations does Pettitt/SNHT also confirm?
    confirmed = {}
    for stn in flagged_list:
        row = pmft_summary[(pmft_summary['index_no'] == stn) & (pmft_summary['variable'] == variable_label)]
        confirmed[stn] = bool(row['significant'].iloc[0]) if not row.empty else False
    confirmed_set = {s for s, ok in confirmed.items() if ok}

    # Step 2: for confirmed stations, is the break isolated from confirmed neighbors?
    rows = []
    for stn in flagged_list:
        is_confirmed = confirmed[stn]
        isolated = None
        decision = 'KEEP'

        if is_confirmed:
            own_events = visual_summary[visual_summary['index_no'] == stn]
            neighbors = matched_stn_dict.get(stn, [])
            dmca_flagged_neighbors = [n for n in neighbors if n in visual_summary['index_no'].unique()]
            isolated = True
            for n in dmca_flagged_neighbors:
                n_events = visual_summary[visual_summary['index_no'] == n]
                for _, oe in own_events.iterrows():
                    for _, ne in n_events.iterrows():
                        overlap = max(oe['start_date'], ne['start_date']) <= min(oe['end_date'], ne['end_date'])
                        if overlap:
                            isolated = False

            decision = 'TRIM' if isolated else 'KEEP'

        rows.append({
            'index_no': stn,
            'variable': variable_label,
            'dmca_flagged': True,
            'pettitt_significant': is_confirmed,
            'isolated': isolated,
            'decision': decision,
        })

    return pd.DataFrame(rows)


removal_plan_tmax = build_removal_plan(tmax_flagged, tmax_matched_stn, dmca_visual_summary, pmft_summary, 'tmax')
removal_plan_tmin = build_removal_plan(tmin_flagged, tmin_matched_stn, dmca_visual_summary, pmft_summary, 'tmin')

removal_plan = pd.concat([removal_plan_tmax, removal_plan_tmin], ignore_index=True)
removal_plan

In [ ]:
# %%
# Apply the plan: for TRIM stations, drop only rows inside that station's
# flagged deviation window(s) from Check 6. Everything else in qc3 is untouched.

def apply_trim(qc_df, visual_summary, removal_plan, variable_label):
    trim_stations = removal_plan[
        (removal_plan['variable'] == variable_label) & (removal_plan['decision'] == 'TRIM')
    ]['index_no'].tolist()

    if not trim_stations:
        print(f"{variable_label}: no stations meet the trim criteria - data unchanged.")
        return qc_df.copy()

    drop_mask = pd.Series(False, index=qc_df.index)
    print(f"{variable_label}: trimming flagged period(s) for {len(trim_stations)} station(s)")

    for stn in trim_stations:
        stn_id_int = int(stn)
        stn_events = visual_summary[visual_summary['index_no'] == stn]
        for _, ev in stn_events.iterrows():
            window_mask = (
                (qc_df['index_no'] == stn_id_int) &
                (qc_df['date'] >= pd.Timestamp(ev['start_date'])) &
                (qc_df['date'] <= pd.Timestamp(ev['end_date']))
            )
            print(f"  station {stn}: {ev['start_date']} -> {ev['end_date']}  ({window_mask.sum()} rows removed)")
            drop_mask |= window_mask

    return qc_df[~drop_mask].reset_index(drop=True)


tmax_qc4 = apply_trim(tmax_qc3, dmca_visual_summary, removal_plan, 'tmax')
tmin_qc4 = apply_trim(tmin_qc3, dmca_visual_summary, removal_plan, 'tmin')

print(f"\nTmax: {len(tmax_qc3) - len(tmax_qc4)} rows removed ({len(tmax_qc4)} remain)")
print(f"Tmin: {len(tmin_qc3) - len(tmin_qc4)} rows removed ({len(tmin_qc4)} remain)")

In [ ]:
# Save the cleaned insitu data
output_folder = r"..\Data_Temp\Insitu\Cleaned"
os.makedirs(output_folder, exist_ok=True)
metadata_subset = metadata[['Index No.', 'Lat', 'Lon', 'Elev']].rename(columns={'Index No.': 'index_no', 'Lat': 'lat', 'Lon': 'long', 'Elev': 'elevation'})
metadata_subset['index_no'] = metadata_subset['index_no'].astype(str)
tmax_qc4['index_no'] = tmax_qc4['index_no'].astype(str)
tmin_qc4['index_no'] = tmin_qc4['index_no'].astype(str)
tmax_qc4 = tmax_qc4.merge(metadata_subset, on='index_no', how='left')
tmin_qc4 = tmin_qc4.merge(metadata_subset, on='index_no', how='left')

tmax_qc4['index_no'] = tmax_qc4['index_no'].astype(int)
tmin_qc4['index_no'] = tmin_qc4['index_no'].astype(int)


tmax_qc4.to_parquet(os.path.join(output_folder, fr"tmax_cleaned.parquet"), index = False)
tmin_qc4.to_parquet(os.path.join(output_folder, fr"tmin_cleaned.parquet"), index = False)

# Total removal stat
print(f"\nTmax: {((len(tmax_raw) - len(tmax_qc4))/len(tmax_raw))*100} % rows removed ")
print(f"Tmin: {((len(tmin_raw) - len(tmin_qc4))/len(tmin_raw)) * 100} % rows removed ")

In [1]:
import pandas as pd
tmax_parquet = pd.read_parquet(r"E:\ClimateData_Temperature\Data\Insitu\Cleaned\tmax_cleaned.parquet")
tmin_parquet = pd.read_parquet(r"E:\ClimateData_Temperature\Data\Insitu\Cleaned\tmin_cleaned.parquet")

tmax_unique_stations = tmax_parquet.drop_duplicates(subset=['index_no', 'lat', 'long'])
tmax_unique_stations.to_csv(r"E:\ClimateData_Temperature\Data\Insitu\Cleaned\tmax_unique_stations.csv")
tmin_unique_stations = tmin_parquet.drop_duplicates(subset=['index_no', 'lat', 'long'])
tmin_unique_stations.to_csv(r"E:\ClimateData_Temperature\Data\Insitu\Cleaned\tmin_unique_stations.csv")